Renad Ebn Alameer
renadebnalameer@gmail.com

**الهدف من المشروع:**

تقديم رؤية شاملاً ومباشرة للإدارة العليا ومديري الفروع لتحسين المبيعات، ومقارنة أداء المبيعات الإلكترونية مع الفروع الميدانية، ومراقبة مستوى رضا العملاء ومعدلات الإرجاع عبر المدن السعودية (الرياض، جدة، الدمام، الخبر).

**Project Objective:**


"To provide executive leadership and store managers with a comprehensive, real-time view of sales performance, enable direct comparisons between e-commerce and brick-and-mortar channels, and monitor customer satisfaction (CSAT) alongside return rates across key Saudi cities (Riyadh, Jeddah, Dammam, and Khobar)."



In [24]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("retail_sales.csv", parse_dates=["date"])
df.head()

,date,store,city,category,channel,units_sold,revenue_sar,returns_units,foot_traffic,csat
0,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,In-Store,52,45097.25,1,950.0,3.98
1,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,Online,12,9323.66,0,NaN,4.00
2,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,In-Store,57,7770.91,2,950.0,4.37
3,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,Online,7,909.04,0,NaN,4.03
4,2025-01-01,Riyadh-Olaya,Riyadh,Home & Kitchen,In-Store,52,11611.75,2,950.0,3.75


In [25]:
import pandas as pd

# 1. تحميل البيانات وتحويل عمود التاريخ
df = pd.read_csv("retail_sales.csv")
df["date"] = pd.to_datetime(df["date"])

# تحويل التاريخ إلى صيغة شهرية (Year-Month)
df["month"] = df["date"].dt.to_period("M")

# تحديد آخر شهر وأحدث 12 شهر في البيانات
latest_month = df["month"].max()
last12 = df[df["date"] > (df["date"].max() - pd.DateOffset(months=12))]

# --- الأرقام الأساسية اللي بتظهر باللوحة ---

# 1) نسبة التبني الرقمي / المبيعات الإلكترونية (آخر شهر)
latest_data = df[df["month"] == latest_month]
online_rev = latest_data[latest_data["channel"] == "Online"]["revenue_sar"].sum()
total_rev = latest_data["revenue_sar"].sum()

online_adoption_pct = round((online_rev / total_rev) * 100, 1)

# 2) عدد المدن تحت هدف رضا العملاء (Target CSAT = 4.0)
TARGET_CSAT = 4.0
by_city_latest = (
    latest_data.groupby("city", as_index=False)["csat"]
    .mean()
)
cities_below_target = int((by_city_latest["csat"] < TARGET_CSAT).sum())

# 3) إجمالي الإيرادات (آخر 12 شهر)
total_revenue = int(last12["revenue_sar"].sum())

# 4) متوسط رضا العملاء (CSAT) لآخر شهر
avg_csat = round(latest_data["csat"].mean(), 2)

# --- طباعة النتائج ---
print(f"Online sales adoption: {online_adoption_pct}%")
print(f"Cities below CSAT target ({TARGET_CSAT}): {cities_below_target}")
print(f"Total revenue (12mo): {total_revenue:,} SAR")
print(f"Average CSAT: {avg_csat}")

Online sales adoption: 26.7%
Cities below CSAT target (4.0): 4
Total revenue (12mo): 148,791,246 SAR
Average CSAT: 3.91


In [26]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. تحميل وتجهيز البيانات
df = pd.read_csv("retail_sales.csv")
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M")

# استخراج بيانات الشهر الأخير والشهر/السنة للتجمعات
latest_month = df["month"].max()
latest_data = df[df["month"] == latest_month]

# --- الحسابات والتجميعات ---

# أ) مؤشرات الصف الأول (KPIs)
total_revenue = df["revenue_sar"].sum()
online_rev = df[df["channel"] == "Online"]["revenue_sar"].sum()
online_share = (online_rev / total_revenue) * 100
avg_csat = df["csat"].mean()
total_units = df["units_sold"].sum()

# ب) اتجاه الإيرادات الشهري (الصف الثاني - يسار)
monthly_trend = (
    df.groupby(df["date"].dt.strftime("%Y-%m"))["revenue_sar"].sum().reset_index()
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# بناء الهيكل الفارغ (Skeleton)
fig = make_subplots(
    rows=3,
    cols=4,
    specs=[
        [
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
            {"type": "domain"},
        ],
        [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
        [{"type": "xy", "colspan": 4}, None, None, None],
    ],
    row_heights=[0.18, 0.42, 0.40],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    subplot_titles=(
        None,
        None,
        None,
        None,
        "National adoption trend (last 12 months)",
        "Bottom 5 regions (latest month)",
        "Users by channel",
    ),
)

# ضبط العنوان وأبعاد اللوحة الفارغة
fig.update_layout(height=650, width=1050, title="Empty layout — just the skeleton")

# عرض الشبكة الفارغة
fig.show()

In [27]:
# 4.1 — إضافة بطاقات KPI (الصف الأول)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=online_adoption_pct,
        number={"suffix": "%", "font": {"size": 32}},
        title={"text": "National Adoption"},
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=cities_below_target,
        number={
            "font": {
                "size": 32,
                "color": "#D55E00" if cities_below_target else "#333",
            }
        },
        title={"text": "Regions Below Target"},
    ),
    row=1,
    col=2,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_revenue,
        number={"valueformat": ",.0f", "font": {"size": 32}},
        title={"text": "Total Users (12mo)"},
    ),
    row=1,
    col=3,
)

fig.add_trace(
    go.Indicator(
        mode="number",
        value=avg_csat,
        number={"font": {"size": 32}},
        title={"text": "Avg. CSAT"},
    ),
    row=1,
    col=4,
)

# تحديث العنوان وارتفاع الشكل للخطوة 4.1
fig.update_layout(height=650, width=1050, title="Step 4.1 — KPI cards added")

# عرض اللوحة بعد إضافة بطاقات الـ KPI
fig.show()

In [28]:
# 4.2 — تجهيز بيانات الاتجاه الشهري (National Trend)
# نلخص نسبة المبيعات الرقمية (Online %) لكل شهر
national_trend = (
    df.groupby("month")
    .apply(
        lambda x: pd.Series(
            {
                "digital_adoption_pct": (
                    x[x["channel"] == "Online"]["revenue_sar"].sum()
                    / x["revenue_sar"].sum()
                )
                * 100
            }
        )
    )
    .reset_index()
)

# تحويل صيغة الشهر إلى نص متوافق مع محور السينات في Plotly
national_trend["month_str"] = national_trend["month"].astype(str)

# إضافة الرسم الخطي في الصف الثاني - العمود الأول
fig.add_trace(
    go.Scatter(
        x=national_trend["month_str"],
        y=national_trend["digital_adoption_pct"],
        mode="lines+markers",
        line=dict(color="#1D9E75", width=2.5),
        showlegend=False,
    ),
    row=2,
    col=1,
)

# تحديث تسميات المحاور
fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="Adoption (%)", row=2, col=1)

# تحديث عنوان اللوحة والتأكد
fig.update_layout(title="Step 4.2 — Trend line added")
fig.show()

/tmp/ipykernel_4232/2653795835.py:5: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [29]:
# 4.3 — تجهيز بيانات أقل المناطق في الشهر الأخير (Bottom Regions)
by_region_latest = (
    latest_data.groupby("city")
    .apply(
        lambda x: pd.Series(
            {
                "digital_adoption_pct": (
                    x[x["channel"] == "Online"]["revenue_sar"].sum()
                    / x["revenue_sar"].sum()
                )
                * 100
            }
        )
    )
    .reset_index()
    .rename(columns={"city": "region"})
)

# ترتيب البيانات وتحديد أقل الفئات
bottom5 = by_region_latest.sort_values("digital_adoption_pct").head(5)

# إضافة الشريط الأفقي في الصف الثاني - العمود الثالث
fig.add_trace(
    go.Bar(
        x=bottom5["digital_adoption_pct"],
        y=bottom5["region"],
        orientation="h",
        marker_color="#D55E00",
        showlegend=False,
    ),
    row=2,
    col=3,
)

# تحديث تسمية المحور
fig.update_xaxes(title_text="Adoption (%)", row=2, col=3)

# تحديث عنوان اللوحة والتأكد
fig.update_layout(title="Step 4.3 — Bottom 5 regions added")
fig.show()

/tmp/ipykernel_4232/3637426532.py:4: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [30]:
# 4.4 — تجهيز بيانات توزيع القنوات (الصف الثالث - عرض كامل)
# نستخدم إجمالي الوحدات المباعة (units_sold) كمقابل لمجموع المستخدمين (unique_users)
by_channel = (
    df.groupby("channel", as_index=False)["units_sold"]
    .sum()
    .rename(columns={"units_sold": "unique_users"})
    .sort_values("unique_users")
)

# إضافة الرسم البياني للأعمدة الأفقية
fig.add_trace(
    go.Bar(
        x=by_channel["unique_users"],
        y=by_channel["channel"],
        orientation="h",
        marker_color="#378ADD",
        showlegend=False,
    ),
    row=3,
    col=1,
)

# تحديث تسمية المحور السيني
fig.update_xaxes(title_text="Total users", row=3, col=1)

# تحديث العنوان النهائي وإظهار اللوحة المكتملة
fig.update_layout(
    title="Step 4.4 — Channel breakdown added — dashboard complete"
)
fig.show()

In [31]:
# 5 — تنظيف العنوان والتنسيق العام (اللمسات النهائية)

fig.update_layout(
    title=dict(
        text="Saudi Retail Sales Dashboard — Store & Category Performance",
        font=dict(size=20, family="Arial"),
        x=0.5,
        xanchor="center",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    height=680,
    width=1080,
    margin=dict(l=40, r=40, t=90, b=40),
)

# تعديل حجم خط العناوين الفرعية (Subplot Titles)
for ann in fig.layout.annotations:
    ann.font = dict(size=13)

# عرض اللوحة بالشكل والتنسيق النهائي المكتمل
fig.show()

In [32]:
fig.write_html("retail_sales_dashboard.html", include_plotlyjs="cdn")
print("Saved: retail_sales_dashboard.html")
print("Open this file in any browser — no Python needed to view it.")

Saved: retail_sales_dashboard.html
Open this file in any browser — no Python needed to view it.
